In [1]:
import os

BASE_PATH = "c:/Users/Revathy/Desktop/WL_Projects/dr_M/dr_m_videos_transcribe"
os.chdir(BASE_PATH)

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import os
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()



c:\Users\Revathy\miniconda3\envs\dr_m_transcribe_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
os.environ["OPENAI_API_BASE"] ="https://api.openai.com/v1"
openai_api_key=os.getenv('OPENAI_API_KEY')

In [4]:
from langchain.docstore.document import Document

def load_transcripts(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            path = os.path.join(folder_path, filename)
            loader = TextLoader(path, encoding="utf-8")
            docs = loader.load()
            # Optionally add video_id as metadata from filename
            for doc in docs:
                doc.metadata["source"] = filename.replace(".txt", "")
            documents.extend(docs)
    return documents


In [5]:
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " "]
    )
    return splitter.split_documents(documents)


In [6]:

def create_vectorstore(chunks):
    # embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)
    embeddings=HuggingFaceEmbeddings(
        model_name="thenlper/gte-base"  # or any other model you prefer
    )
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_store"
    )
    vectorstore.persist()
    return vectorstore


In [7]:
def create_qa_chain(vectorstore):
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
    llm = ChatOpenAI(openai_api_key=openai_api_key, model="gpt-4", temperature=0.3)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True
    )
    return qa_chain


In [8]:
def answer_question(chain, question):
    result = chain({"query": question})
    print("\n📌 Answer:")
    print(result["result"])
    
    print("\n📄 Sources:")
    for doc in result["source_documents"]:
        print(f"- {doc.metadata['source']}")


In [9]:
folder_path = "data/whisper_hindi"  # Folder where your .txt files are
documents = load_transcripts(folder_path)


In [10]:
chunks = split_documents(documents)



In [11]:
vectorstore = create_vectorstore(chunks)


C:\Users\Revathy\AppData\Local\Temp\ipykernel_11008\2145837910.py:11: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [ ]:
qa_chain = create_qa_chain(vectorstore)

while True:
    q = input("\n❓ Ask a question (or type 'exit'): ")
    if q.lower() in ['exit', 'quit']:
        break
    answer_question(qa_chain, q)

C:\Users\Revathy\AppData\Local\Temp\ipykernel_11008\641092498.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(openai_api_key=openai_api_key, model="gpt-4", temperature=0.3)
C:\Users\Revathy\AppData\Local\Temp\ipykernel_11008\3424240010.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain({"query": question})



📌 Answer:
The red flags in a relationship are:

1. Mixed signals: They say one thing but do another. Love should feel steady, not like guessing the next move in a game of chess.
2. One-sided effort: If you're always the one trying to connect, it's not a partnership, it's one sided. If they are emotionally unavailable, it's a red flag.
3. Constant uncertainty: You're always questioning their intentions or your place in their life.
4. Disregard for your emotions: If they make you feel like your emotions are too much, they're not listening. You deserve someone who values your feelings.
5. Holding on to hope, not reality: You're in love with their potential, not the present. A healthy relationship is about what is, not what could be.

📄 Sources:
- z9D6UmI4pBU
- z9D6UmI4pBU

📌 Answer:
I'm sorry, but I can't provide the help you're looking for.

📄 Sources:
- O7NIn7WaTxY
- O7NIn7WaTxY

📌 Answer:
I'm sorry, but I can't assist with that.

📄 Sources:
- O7NIn7WaTxY
- O7NIn7WaTxY

📌 Answer:
معذرت